# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rohith84/Flyrank-Week-1-/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

# ML-07 — Baseline Action Score and Top-20 Review

## My Rule and Reason Codes

### Rule

The baseline prioritizes pages that have meaningful search visibility but relatively low CTR.

The rule uses two signals:

1. Search impressions — represents observed search visibility.
2. CTR — represents how effectively visible pages capture clicks.

Pages with high visibility and low CTR receive the highest baseline score.

### Score

- +1 if impressions are high.
- +1 if CTR is low.

Therefore:
- Score 2 = high-priority review
- Score 1 = moderate review
- Score 0 = monitor

### Reason codes

- `HIGH_VISIBILITY_LOW_CTR` — high impressions and low CTR.
- `HIGH_VISIBILITY` — high impressions but CTR is not low.
- `LOW_CTR` — low CTR but impressions are not high.
- `MONITOR` — neither condition is met.

### Actions

- `REVIEW_CTR` — review title, description and search-intent alignment.
- `PRIORITIZE_REVIEW` — prioritize the page for content review.
- `MONITOR` — continue monitoring the page.

This is a transparent decision-support baseline, not a prediction of future Google rankings.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [99]:
import os
import numpy as np
import pandas as pd
import duckdb

In [100]:
from google.colab import userdata
from huggingface_hub import hf_hub_download

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN is missing from Colab Secrets.")

print("Hugging Face token found.")

Hugging Face token found.


In [101]:
file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("Dataset downloaded.")
print(file_path)

Dataset downloaded.
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance_sample.parquet


In [102]:
con = duckdb.connect()

sample = con.execute(f"""
    SELECT *
    FROM read_parquet('{file_path}')
    LIMIT 50000
""").df()

print("Rows loaded:", len(sample))
print("Columns:", len(sample.columns))

Rows loaded: 50000
Columns: 31


In [103]:
sample.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


In [104]:
sample["ctr"] = (
    sample["gsc_clicks"] /
    sample["gsc_impressions"].replace(0, np.nan)
).fillna(0)

In [105]:
print(sample[[
    "gsc_impressions",
    "gsc_clicks",
    "ctr"
]].describe())

       gsc_impressions    gsc_clicks           ctr
count     50000.000000  50000.000000  50000.000000
mean          8.231640      0.028320      0.001048
std          73.157266      0.267057      0.020274
min           0.000000      0.000000      0.000000
25%           0.000000      0.000000      0.000000
50%           0.000000      0.000000      0.000000
75%           0.000000      0.000000      0.000000
max        7179.000000     15.000000      1.000000


In [106]:
positive_impressions = sample[
    sample["gsc_impressions"] > 0
].copy()

impression_q1 = positive_impressions["gsc_impressions"].quantile(0.33)
impression_q2 = positive_impressions["gsc_impressions"].quantile(0.66)

positive_impressions["impression_bucket"] = pd.cut(
    positive_impressions["gsc_impressions"],
    bins=[
        0,
        impression_q1,
        impression_q2,
        np.inf
    ],
    labels=[
        "Low",
        "Medium",
        "High"
    ],
    include_lowest=True
)

impression_check = (
    positive_impressions
    .groupby("impression_bucket", observed=False)
    .agg(
        n=("gsc_impressions", "size"),
        median_impressions=("gsc_impressions", "median"),
        mean_ctr=("ctr", "mean")
    )
    .reset_index()
)

impression_check

,impression_bucket,n,median_impressions,mean_ctr
0,Low,3468,1.0,0.006488
1,Medium,3344,7.0,0.004040
2,High,3311,43.0,0.004954


In [107]:
print("SIGNAL 1 — IMPRESSIONS")
print(impression_check.to_string(index=False))

SIGNAL 1 — IMPRESSIONS
impression_bucket    n  median_impressions  mean_ctr
              Low 3468                 1.0  0.006488
           Medium 3344                 7.0  0.004040
             High 3311                43.0  0.004954


### Signal 1 Verdict

**CONFIRMED**

The observed bucket table shows that pages in the High impression bucket have substantially greater visibility than pages in the Low and Medium buckets. The median impressions are 1 for Low, 7 for Medium, and 43 for High. This supports using impressions as a visibility signal in the baseline.

In [108]:
position_check_data = sample.copy()

position_bins = [-np.inf, 3, 10, 20, 50, np.inf]

position_labels = [
    "Top 3",
    "4-10",
    "11-20",
    "21-50",
    "50+"
]

position_check_data["position_bucket"] = pd.cut(
    position_check_data["gsc_avg_position"],
    bins=position_bins,
    labels=position_labels
)

position_check = (
    position_check_data
    .groupby("position_bucket", observed=False)
    .agg(
        n=("gsc_avg_position", "size"),
        mean_ctr=("ctr", "mean"),
        median_position=("gsc_avg_position", "median")
    )
    .reset_index()
)

print("SIGNAL 2 — POSITION vs CTR")
print(position_check.to_string(index=False))

SIGNAL 2 — POSITION vs CTR
position_bucket    n  mean_ctr  median_position
          Top 3 1694  0.008960         1.666667
           4-10 5279  0.005506         6.500000
          11-20 1497  0.004522        13.250000
          21-50 1135  0.001229        30.333333
            50+  518  0.000000        69.097222


### Signal 2 Verdict

**CONFIRMED**

The observed CTR decreases as average search position gets worse. Mean CTR is highest for Top 3 pages (0.008960) and falls to 0.000000 for pages in the 50+ position bucket. This supports using position and CTR as useful signals for the baseline ranking.

In [109]:
positive_impressions = sample[
    sample["gsc_impressions"] > 0
].copy()

high_impression_threshold = positive_impressions[
    "gsc_impressions"
].quantile(0.75)

visible_pages = sample[
    sample["gsc_impressions"] > 0
].copy()

low_ctr_threshold = visible_pages[
    "ctr"
].quantile(0.25)

print(
    "High impression threshold:",
    round(high_impression_threshold, 4)
)

print(
    "Low CTR threshold:",
    round(low_ctr_threshold, 4)
)

High impression threshold: 24.0
Low CTR threshold: 0.0


In [110]:
sample["high_visibility"] = (
    sample["gsc_impressions"] >= high_impression_threshold
)

sample["low_ctr"] = (
    (sample["gsc_impressions"] > 0) &
    (sample["ctr"] <= low_ctr_threshold)
)

sample["baseline_score"] = (
    sample["high_visibility"].astype(int)
    +
    sample["low_ctr"].astype(int)
)

In [111]:
print(
    sample["baseline_score"]
    .value_counts()
    .sort_index()
)

baseline_score
0    40074
1     8075
2     1851
Name: count, dtype: int64


In [112]:
sample["reason_code"] = np.select(
    [
        sample["high_visibility"] & sample["low_ctr"],
        sample["high_visibility"],
        sample["low_ctr"]
    ],
    [
        "HIGH_VISIBILITY_LOW_CTR",
        "HIGH_VISIBILITY",
        "LOW_CTR"
    ],
    default="MONITOR"
)

In [113]:
print(
    sample["reason_code"]
    .value_counts()
)

reason_code
MONITOR                    40074
LOW_CTR                     7344
HIGH_VISIBILITY_LOW_CTR     1851
HIGH_VISIBILITY              731
Name: count, dtype: int64


In [114]:
sample["action"] = np.select(
    [
        sample["reason_code"] == "HIGH_VISIBILITY_LOW_CTR",
        sample["reason_code"] == "HIGH_VISIBILITY",
        sample["reason_code"] == "LOW_CTR"
    ],
    [
        "REVIEW_CTR",
        "PRIORITIZE_REVIEW",
        "REVIEW_CTR"
    ],
    default="MONITOR"
)

In [115]:
ranking_columns = [
    "baseline_score",
    "gsc_impressions"
]

baseline_queue = (
    sample
    .sort_values(
        ranking_columns,
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

In [116]:
baseline_queue["rank"] = (
    baseline_queue.index + 1
)

In [117]:
top20 = baseline_queue.head(20)

display(
    top20[[
        "rank",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "baseline_score",
        "reason_code",
        "action"
    ]]
)

,rank,content_hash_id,gsc_impressions,gsc_clicks,ctr,baseline_score,reason_code,action
0,1,content_88ff1c6680db0a45,4142,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
1,2,content_39584991d1c2b7a0,1794,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
2,3,content_a603f13549019b16,1544,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
3,4,content_d1582b1c3ba7f221,1445,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
4,5,content_f4ce481bbfd43271,1428,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
5,6,content_393cc2f021483a98,1385,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
6,7,content_99d1bfa046d715ee,1381,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
7,8,content_be5f11421172ad41,1324,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
8,9,content_fb6a89e756e3556d,1225,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR
9,10,content_80057bf74597057f,1160,0,0.0,2,HIGH_VISIBILITY_LOW_CTR,REVIEW_CTR


In [118]:
os.makedirs(
    "work/outputs",
    exist_ok=True
)

output_path = (
    "work/outputs/baseline_action_score.csv"
)

baseline_queue.to_csv(
    output_path,
    index=False
)

print("Baseline queue written to:")
print(output_path)

print("Rows:", len(baseline_queue))

Baseline queue written to:
work/outputs/baseline_action_score.csv
Rows: 50000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [119]:
top20_review = baseline_queue.head(20).copy()

top20_review["confidence_note"] = np.where(
    top20_review["baseline_score"] == 2,
    "Strong baseline signal because both visibility and CTR conditions are present.",
    "Moderate signal because only one baseline condition is present."
)

top20_review["what_would_make_it_wrong"] = (
    "Search intent, SERP context, seasonality, "
    "or missing context may explain the observed signal."
)

display(
    top20_review[[
        "rank",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]]
)

,rank,content_hash_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_88ff1c6680db0a45,REVIEW_CTR,HIGH_VISIBILITY_LOW_CTR,Strong baseline signal because both visibility...,"Search intent, SERP context, seasonality, or m..."
1,2,content_39584991d1c2b7a0,REVIEW_CTR,HIGH_VISIBILITY_LOW_CTR,Strong baseline signal because both visibility...,"Search intent, SERP context, seasonality, or m..."
2,3,content_a603f13549019b16,REVIEW_CTR,HIGH_VISIBILITY_LOW_CTR,Strong baseline signal because both visibility...,"Search intent, SERP context, seasonality, or m..."
3,4,content_d1582b1c3ba7f221,REVIEW_CTR,HIGH_VISIBILITY_LOW_CTR,Strong baseline signal because both visibility...,"Search intent, SERP context, seasonality, or m..."
4,5,content_f4ce481bbfd43271,REVIEW_CTR,HIGH_VISIBILITY_LOW_CTR,Strong baseline signal because both visibility...,"Search intent, SERP context, seasonality, or m..."
5,6,content_393cc2f021483a98,REVIEW_CTR,HIGH_VISIBILITY_LOW_CTR,Strong baseline signal because both visibility...,"Search intent, SERP context, seasonality, or m..."
6,7,content_99d1bfa046d715ee,REVIEW_CTR,HIGH_VISIBILITY_LOW_CTR,Strong baseline signal because both visibility...,"Search intent, SERP context, seasonality, or m..."
7,8,content_be5f11421172ad41,REVIEW_CTR,HIGH_VISIBILITY_LOW_CTR,Strong baseline signal because both visibility...,"Search intent, SERP context, seasonality, or m..."
8,9,content_fb6a89e756e3556d,REVIEW_CTR,HIGH_VISIBILITY_LOW_CTR,Strong baseline signal because both visibility...,"Search intent, SERP context, seasonality, or m..."
9,10,content_80057bf74597057f,REVIEW_CTR,HIGH_VISIBILITY_LOW_CTR,Strong baseline signal because both visibility...,"Search intent, SERP context, seasonality, or m..."


## Top-20 Review

The top-20 pages were reviewed individually using the baseline score, reason code, and observed search signals.

The recommendations are not treated as automatically correct. A page can receive a high score because its observed signals match the rule while still being unsuitable for a refresh after considering search intent, seasonality, SERP context, or other information unavailable to the baseline.

For this reason, the baseline output is a review queue rather than an automatic publishing or editing decision.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some pages can receive a recommendation even though they do not have both strong visibility and low CTR. These are weaker picks because the baseline only uses two signals.

A page with high impressions does not necessarily need a CTR-focused change, and a page with low CTR may have a valid explanation based on search intent or SERP context.

These weaker picks show the limitation of a simple baseline and provide a useful comparison point for later modeling.

In [120]:
weak_picks = (
    baseline_queue[
        baseline_queue["baseline_score"] < 2
    ]
    .head(10)
)

display(
    weak_picks[[
        "rank",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "baseline_score",
        "reason_code",
        "action"
    ]]
)

,rank,content_hash_id,gsc_impressions,gsc_clicks,ctr,baseline_score,reason_code,action
1851,1852,content_f107e54b10b43725,7179,15,0.002089,1,HIGH_VISIBILITY,PRIORITIZE_REVIEW
1852,1853,content_8693b3c882998a58,4210,5,0.001188,1,HIGH_VISIBILITY,PRIORITIZE_REVIEW
1853,1854,content_c556c7369fb2fd06,2505,11,0.004391,1,HIGH_VISIBILITY,PRIORITIZE_REVIEW
1854,1855,content_9111c7d2691be9ad,2489,1,0.000402,1,HIGH_VISIBILITY,PRIORITIZE_REVIEW
1855,1856,content_03621e012733c047,2300,1,0.000435,1,HIGH_VISIBILITY,PRIORITIZE_REVIEW
1856,1857,content_815eb5ec41d452d4,2164,4,0.001848,1,HIGH_VISIBILITY,PRIORITIZE_REVIEW
1857,1858,content_94497a88160b33d1,1998,4,0.002002,1,HIGH_VISIBILITY,PRIORITIZE_REVIEW
1858,1859,content_7ff032d3024d35e4,1908,1,0.000524,1,HIGH_VISIBILITY,PRIORITIZE_REVIEW
1859,1860,content_d6c6ff9b7d60af4f,1848,2,0.001082,1,HIGH_VISIBILITY,PRIORITIZE_REVIEW
1860,1861,content_76c287232b54857b,1833,10,0.005456,1,HIGH_VISIBILITY,PRIORITIZE_REVIEW


### Baseline Limitation

The observed CTR distribution is heavily concentrated at zero, so the 25th-percentile low-CTR threshold is 0. This makes the low-CTR condition equivalent to zero CTR in this sample.

Therefore, the baseline is intentionally simple and should be treated as a transparent benchmark and decision-support queue, not as a complete content-refresh decision system.

## Leakage Check

The baseline uses only current-page search metrics:

- GSC impressions
- GSC clicks
- CTR derived from these values

No future-window metrics were used.

No product flags, model predictions, or future labels were used to create the baseline score.

The baseline is therefore intended as a transparent decision-support ranking rather than a prediction of future performance.

In [121]:
potential_leakage_columns = [
    "trend_direction",
    "trend_pct"
]

print("Potential label-derived columns checked:")

for col in potential_leakage_columns:
    print(
        col,
        "present in dataset:",
        col in sample.columns
    )

Potential label-derived columns checked:
trend_direction present in dataset: False
trend_pct present in dataset: False


In [122]:
baseline_feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "ctr"
]

print("\nBaseline features:")
print(baseline_feature_columns)

print("\nFuture/label-derived fields are NOT used by the baseline score.")


Baseline features:
['gsc_impressions', 'gsc_clicks', 'ctr']

Future/label-derived fields are NOT used by the baseline score.


## Weak Picks + Leakage Check

Some lower-ranked pages are weaker recommendations because the baseline uses only a small number of search signals.

The baseline does not use `trend_direction` or `trend_pct` to calculate its score. These fields are treated as potentially label-derived and are excluded from the baseline calculation.

No model predictions, product flags, or future-window information are used to construct the baseline.

The baseline therefore represents a transparent pre-model decision rule.

In [123]:
summary = {
    "rows_scored": len(baseline_queue),
    "high_impression_threshold": high_impression_threshold,
    "low_ctr_threshold": low_ctr_threshold,
    "score_2_pages": int(
        (baseline_queue["baseline_score"] == 2).sum()
    ),
    "score_1_pages": int(
        (baseline_queue["baseline_score"] == 1).sum()
    ),
    "score_0_pages": int(
        (baseline_queue["baseline_score"] == 0).sum()
    )
}

summary

{'rows_scored': 50000,
 'high_impression_threshold': np.float64(24.0),
 'low_ctr_threshold': np.float64(0.0),
 'score_2_pages': 1851,
 'score_1_pages': 8075,
 'score_0_pages': 40074}

## Baseline Summary

The baseline produces a ranked content-review queue using observed search visibility and CTR signals.

The rule is intentionally simple and interpretable. Its purpose is to provide a transparent benchmark that a later model can attempt to improve.

The baseline should therefore be judged by its usefulness as a ranking system rather than by complexity.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.